# Imports

In [1]:
# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# Paths
import os
data_path = os.path.join('..', 'data')

# Translate nucleotide sequences to amino acid sequences
from Bio.Seq import Seq

# Load Data

In [2]:
# Load genes human with expression data
genes_human_expression_path = os.path.join(data_path, 'genes_human_with_expression.pkl')
df = pd.read_pickle(genes_human_expression_path)

# Display the first few rows of the dataframe
df.head()

,seq,gene,chr,strand,coord_start,coord_end,canonical,ENC,transcript,exp_T,exp_HEK293,exp_U2OS,cARS_T,cARS_HEK293,cARS_U2OS,median_float,tissueSiteDetailId
0,ATGGCGTCCCCGTCTCGGAGACTGCAGACTAAACCAGTCATTACTT...,ENSG00000000003,X,-,100627107,100636806,True,45.363981,ENST00000373020,6.113815,9.838259,14.0,0.0,0.0,0.0,"[25.280000686645508, 22.770000457763672, 15.26...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
1,ATGCTAAAACTGTATGCAATGTTTCTGACTCTCGTTTTTTTGGTCG...,ENSG00000000003,X,-,100627108,100637104,False,38.175250,ENST00000612152,6.113815,9.838259,14.0,0.0,0.0,0.0,"[1.0399999618530273, 0.8700000047683716, 0.485...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
2,ATGGCAAAGAATCCTCCAGAGAATTGTGAAGACTGTCACATTCTAA...,ENSG00000000005,X,+,100584935,100599885,True,45.428790,ENST00000373031,3.349926,2.617150,0.0,0.0,0.0,0.0,"[15.739999771118164, 6.639999866485596, 0.0, 0...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
3,ATGGCCTCCTTGGAAGTCAGTCGTAGTCCTCGCAGGTCTCGGCGGG...,ENSG00000000419,20,-,50934866,50958550,False,46.773340,ENST00000466152,11.399582,10.070030,213.0,0.0,0.0,0.0,"[1.0700000524520874, 0.9200000166893005, 0.699...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
4,ATGGCCTCCTTGGAAGTCAGTCGTAGTCCTCGCAGGTCTCGGCGGG...,ENSG00000000419,20,-,50934866,50958555,False,47.072886,ENST00000371582,11.399582,10.070030,213.0,0.0,0.0,0.0,"[1.3700000047683716, 1.600000023841858, 1.5449...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80225 entries, 0 to 80224
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   seq                 80225 non-null  object 
 1   gene                80225 non-null  object 
 2   chr                 80225 non-null  object 
 3   strand              80225 non-null  object 
 4   coord_start         80225 non-null  int64  
 5   coord_end           80225 non-null  int64  
 6   canonical           80225 non-null  bool   
 7   ENC                 80225 non-null  float64
 8   transcript          80225 non-null  object 
 9   exp_T               80225 non-null  float64
 10  exp_HEK293          80225 non-null  float64
 11  exp_U2OS            80225 non-null  float64
 12  cARS_T              80225 non-null  float64
 13  cARS_HEK293         80225 non-null  float64
 14  cARS_U2OS           80225 non-null  float64
 15  median_float        63472 non-null  object 
 16  tiss

# Clean Data

**Drop unused columns:**

In [4]:
df.drop(columns=['cARS_HEK293', 'cARS_T', 'exp_T', 'exp_U2OS', 'exp_HEK293', 'cARS_U2OS'], inplace=True)

**Drop non-multiple-of-three sequences (truncated):**

In [5]:
non_multiple_of_three = (df['seq'].map(len) % 3 != 0)
non_multiple_of_three_indices = df.index[non_multiple_of_three]

print(f"* Dropping {len(non_multiple_of_three_indices)} sequences...")
print(df.loc[non_multiple_of_three_indices][['gene', 'transcript']])
print("sequence lengths: ", df.loc[non_multiple_of_three_indices]['seq'].map(len).values, end=' ')

df.drop(non_multiple_of_three_indices, inplace=True)
df.reset_index(inplace=True, drop=True)

* Dropping 7 sequences...
                  gene       transcript
46558  ENSG00000155657  ENST00000342992
46559  ENSG00000155657  ENST00000460472
46560  ENSG00000155657  ENST00000589042
46561  ENSG00000155657  ENST00000591111
46562  ENSG00000155657  ENST00000342175
46563  ENSG00000155657  ENST00000359218
64533  ENSG00000181143  ENST00000397910
sequence lengths:  [32765 32765 32765 32765 32765 32765 32765] 

## Add amino acid column

In [6]:
df['amino_acid_seq'] = df['seq'].apply(lambda x: str(Seq(x).translate()))
df.head()

,seq,gene,chr,strand,coord_start,coord_end,canonical,ENC,transcript,median_float,tissueSiteDetailId,amino_acid_seq
0,ATGGCGTCCCCGTCTCGGAGACTGCAGACTAAACCAGTCATTACTT...,ENSG00000000003,X,-,100627107,100636806,True,45.363981,ENST00000373020,"[25.280000686645508, 22.770000457763672, 15.26...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MASPSRRLQTKPVITCFKSVLLIYTFIFWITGVILLAVGIWGKVSL...
1,ATGCTAAAACTGTATGCAATGTTTCTGACTCTCGTTTTTTTGGTCG...,ENSG00000000003,X,-,100627108,100637104,False,38.175250,ENST00000612152,"[1.0399999618530273, 0.8700000047683716, 0.485...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MLKLYAMFLTLVFLVELVAAIVGFVFRHEIKNSFKNNYEKALKQYN...
2,ATGGCAAAGAATCCTCCAGAGAATTGTGAAGACTGTCACATTCTAA...,ENSG00000000005,X,+,100584935,100599885,True,45.428790,ENST00000373031,"[15.739999771118164, 6.639999866485596, 0.0, 0...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MAKNPPENCEDCHILNAEAFKSKKICKSLKICGLVFGILALTLIVL...
3,ATGGCCTCCTTGGAAGTCAGTCGTAGTCCTCGCAGGTCTCGGCGGG...,ENSG00000000419,20,-,50934866,50958550,False,46.773340,ENST00000466152,"[1.0700000524520874, 0.9200000166893005, 0.699...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MASLEVSRSPRRSRRELEVRSPRQNKYSVLLPTYNERENLPLIVWL...
4,ATGGCCTCCTTGGAAGTCAGTCGTAGTCCTCGCAGGTCTCGGCGGG...,ENSG00000000419,20,-,50934866,50958555,False,47.072886,ENST00000371582,"[1.3700000047683716, 1.600000023841858, 1.5449...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MASLEVSRSPRRSRRELEVRSPRQNKYSVLLPTYNERENLPLIVWL...


## Drop non-stop sequences

In [7]:
# Drop rows where 'amino_acid_seq' doens't end with '*'
no_stop_rows = df[~df['amino_acid_seq'].str.endswith('*')].index

print(f"* Dropping {len(no_stop_rows)} sequences...")
df.drop(no_stop_rows, inplace=True)

* Dropping 0 sequences...


## Drop early-stop sequences
**(selenoproteins but CD-HIT won't accept them)**

In [8]:
# Drop rows where 'amino_acid_seq' has an early '*'
early_stop_rows = df[df['amino_acid_seq'].str[1:-1].str.contains('*', regex=False)]
early_stop_rows

,seq,gene,chr,strand,coord_start,coord_end,canonical,ENC,transcript,median_float,tissueSiteDetailId,amino_acid_seq
1897,ATGTCTGAACCAATCAGAGTCCTTGTGACTGGAGCAGCTGGTCAAA...,ENSG00000014641,2,+,63588962,63607197,False,47.442368,ENST00000539945,"[29.65999984741211, 19.90999984741211, 36.5699...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MSEPIRVLVTGAAGQIAYSLLYSIGNGSVFGKDQPIILVLLDITPM...
6225,ATGGCCGTATACAGGGCAGCGCTCGGGGCTTCGCTCGCGGCTGCCC...,ENSG00000073169,22,+,50201010,50217616,True,36.467101,ENST00000380903,"[4.519999980926514, 4.03000020980835, 5.420000...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MAVYRAALGASLAAARLLPLGRCSPSPAPRSTLSGAAMEPAPRWLA...
8218,ATGGACTCCCCGATCCAGATCTTCCGCGGGGAGCCGGGCCCTACCT...,ENSG00000082556,8,-,53225723,53251637,False,43.620557,ENST00000673285,NaN,NaN,MDSPIQIFRGEPGPTCAPSACLPPNSSAWFPGWAEPDSNGSAGSED...
10396,ATGGAAGCGGGACCCTCGGGAGCAGCTGCGGGCGCTTACCTGCCCC...,ENSG00000092847,1,+,35883208,35930532,False,47.622209,ENST00000674426,NaN,NaN,MEAGPSGAAAGAYLPPLQQVFQAPRRPGIGTVGKPIKLLANYFEVD...
11183,ATGGGCCTAGAGCTGTTTCTTGACCTGGTGTCCCAGCCCAGCCGCG...,ENSG00000099984,22,+,23980057,23983710,False,40.943182,ENST00000402588,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MGLELFLDLVSQPSRAVYIFAKKNGIPLELRTVDLVKGQHKSKEFL...
...,...,...,...,...,...,...,...,...,...,...,...,...
78227,ATGAATAACTCACAGATATCTACTGTGACGCAGTTTGTGTTGTTGG...,ENSG00000258806,14,+,20229401,20230346,True,46.545959,ENST00000553765,"[0.11999999731779099, 0.09000000357627869, 0.0...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MNNSQISTVTQFVLLGFPGPWKIQIIFFSMILLVYIFTLTGNMAII...
78363,ATGAACAATGTAACAGAATTCATCCTGCTGGGCCTCACTCACAATC...,ENSG00000260811,11,+,50032873,50033794,True,44.979890,ENST00000568934,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MNNVTEFILLGLTHNPELQKFLFVMFLITYLITLAGNLLISVIIFI...
78636,ATGCTGCCCCCTTGGACCCTCGGCCTTCTCCTGCTGGCCACAGTCA...,ENSG00000266200,10,+,116620952,116645143,True,52.899051,ENST00000579578,"[0.0, 0.0, 0.07000000029802322, 0.0, 0.0, 0.0,...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MLPPWTLGLLLLATVRGKEVCYGQLGCFSDEKPWAGTLQRPVKLLP...
79353,ATGGCCGAGCTGCCCCACAGGATCATCAAGGAAACCCAGCGTTTGC...,ENSG00000276380,X,+,143884070,143885255,True,44.567080,ENST00000618570,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu...",MAELPHRIIKETQRLLAEPVPGIKAEPDESNARYFHVVIAGESKDS...


In [9]:
print(f"* Dropping {len(early_stop_rows)} sequences...")
df.drop(early_stop_rows.index, inplace=True)

* Dropping 123 sequences...


## Drop sequences that don't start with Methionine (M)

In [10]:
no_start_rows = df[df['amino_acid_seq'].str[0] != 'M']
print(f"* Dropping {len(no_start_rows)} sequences...")
df.drop(no_start_rows.index, inplace=True)

* Dropping 0 sequences...


## Drop duplicate rows

In [11]:
df['tissueSiteDetailId'] = df['tissueSiteDetailId'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
df['median_float'] = df['median_float'].apply(lambda x: tuple(x) if isinstance(x, list) else x)

duplicated_rows = df[df.duplicated(keep=False)]
df.drop(duplicated_rows.index, inplace=True)

In [12]:
df[df.duplicated(subset=['seq'], keep=False)].groupby('gene').apply(lambda x: x, include_groups=False)

seq chr  \
gene                                                                           
ENSG00000000457 10     ATGGGATCAGAGAACAGTGCTTTAAAGAGCTATACACTGAGAGAAC...   1   
                11     ATGGGATCAGAGAACAGTGCTTTAAAGAGCTATACACTGAGAGAAC...   1   
ENSG00000000460 13     ATGTTTTTACCTCATATGAACCACCTGACATTGGAACAGACTTTCT...   1   
                15     ATGTCTCAGGAAGGTGCGGTCCCAGCTAGCGCGGTTCCCCTGGAAG...   1   
                16     ATGTCTCAGGAAGGTGCGGTCCCAGCTAGCGCGGTTCCCCTGGAAG...   1   
...                                                                  ...  ..   
ENSG00000291237 80204  ATGTTGAGCCGGGCAGTGTGCGGCACCAGCAGGCAGCTGGCTCCGG...   6   
ENSG00000291239 80206  ATGACATCTCAATCTTCAGTGATCAGCAATAGCTGTGTGACAATGG...  19   
ENSG00000291317 80215  ATGGCCCCCAAGCCGGGGGCCGAGTGGAGCACAGCCCTGTCCCATC...   8   
                80216  ATGGCCCCCAAGCCGGGGGCCGAGTGGAGCACAGCCCTGTCCCATC...   8   
                80217  ATGGCCCCCAAGCCGGGGGCCGAGTGGAGCACAGCCCTGTCCCATC...   8   

                      strand  coord_start  coord_end  canonical        ENC  \
gene                                                                         
ENSG00000000457 10         -    169853073  169888888      False  50.356737   
                11         -    169853073  169893959      False  50.356737   
ENSG00000000460 13         +    169795039  169854080       True  48.094044   
                15         +    169795042  169807837      False  36.142080   
                16         +    169795078  169821719      False  36.142080   
...                      ...          ...        ...        ...        ...   
ENSG00000291237 80204      -    159682164  159693278      False  45.751546   
ENSG00000291239 80206      +     36850857   36998682       True  46.972572   
ENSG00000291317 80215      -    144463816  144465489       True  39.257905   
                80216      -    144463816  144465648      False  39.257905   
                80217      -    144463824  144465492      False  39.257905   

                            transcript  \
gene                                     
ENSG00000000457 10     ENST00000367770   
                11     ENST00000367772   
ENSG00000000460 13     ENST00000359326   
                15     ENST00000481744   
                16     ENST00000466580   
...                                ...   
ENSG00000291237 80204  ENST00000337404   
ENSG00000291239 80206  ENST00000706165   
ENSG00000291317 80215  ENST00000403000   
                80216  ENST00000424149   
                80217  ENST00000306145   

                                                            median_float  \
gene                                                                       
ENSG00000000457 10     (1.809999942779541, 1.25, 0.8799999952316284, ...   
                11     (1.4600000381469727, 1.2400000095367432, 1.200...   
ENSG00000000460 13     (0.23999999463558197, 0.25, 0.2800000011920929...   
                15     (0.0, 0.0, 0.0, 0.0, 0.0, 0.09000000357627869,...   
                16     (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
...                                                                  ...   
ENSG00000291237 80204                                                NaN   
ENSG00000291239 80206                                                NaN   
ENSG00000291317 80215                                                NaN   
                80216                                                NaN   
                80217                                                NaN   

                                                      tissueSiteDetailId  \
gene                                                                       
ENSG00000000457 10     (Adipose_Subcutaneous, Adipose_Visceral_Omentu...   
                11     (Adipose_Subcutaneous, Adipose_Visceral_Omentu...   
ENSG00000000460 13     (Adipose_Subcutaneous, Adipose_Visceral_Omentu...   
                15     (Adipose_Subcutaneous, Adipose_Visceral_Omentu...   
                16   

# Save Data

In [13]:
# Define the path for the cleaned data
cleaned_data_path = os.path.join(data_path, 'genes_human_cleaned.pkl')

# Save the dataframe to the specified path
df.to_pickle(cleaned_data_path)